In [773]:
import os
import time
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from collections import deque
from ultralytics import YOLO
import pandas as pd
import csv

# ------------------------------------------------------------
# ⚙️ CONFIGURATION
# ------------------------------------------------------------
GT_FILE = "/home1/jtt_1/mtp/okutama-action/TrainSetVideos/Labels/SingleActionLabels/3840x2160/2.2.4.txt"
VIDEO_PATH = "/home1/jtt_1/mtp/okutama-action/TrainSetVideos/Drone2/Noon/2.2.4.mp4"
VIDEO_NAME = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
OUTPUT_DIR = f"/home1/jtt_1/mtp/outputs-with-deep-sort-tracking-single/{VIDEO_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# END

CONFIG = {
    "VIDEO_PATH": VIDEO_PATH,
    "VIDEO_NAME": VIDEO_NAME,
    "OUTPUT_DIR": OUTPUT_DIR,

    # YOLO
    "YOLO_MODEL_PATH": "/home1/jtt_1/mtp/best-models/best.pt",
    "CONF_THRESH": 0.70,
    "DETECT_EVERY_N_FRAMES": 1,

    # Tracking
    "OUT_OF_FRAME_FRACTION": 0.9,
    "MAX_AGE": 10,

    # TimeSformer
    "TSF_MODEL_PATH": "/home1/jtt_1/mtp/timesformer_base_patch16_224_k400",
    "TSF_CHECKPOINT": "/home1/jtt_1/mtp/best-models/best_mAP_epoch_23.pth",
    "NUM_FRAMES": 16,
    "TOP_K": 1,

    # Evaluation
    "IOU_THRESH": 0.3,

    # Runtime
    "TRAIN_CSV": "/home1/jtt_1/mtp/timesformer-person-dataset/train_person_labels.csv",
    "TEST_MODE": True,
    "TEST_MAX_FRAMES": 100,
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",

    # Output
    "RESULTS_CSV": f"{OUTPUT_DIR}/{VIDEO_NAME}_tracking.csv",
}

print(f"[INFO] Processing video: {CONFIG['VIDEO_NAME']}")
print(f"[INFO] Using device: {CONFIG['DEVICE']}")
print(f"[INFO] Output folder: {CONFIG['OUTPUT_DIR']}")

[INFO] Processing video: 2.2.4
[INFO] Using device: cuda
[INFO] Output folder: /home1/jtt_1/mtp/outputs-with-deep-sort-tracking-single/2.2.4


In [774]:
cap = cv2.VideoCapture(CONFIG["VIDEO_PATH"])

if not cap.isOpened():
    raise RuntimeError(f"Could not open video file: {CONFIG['VIDEO_PATH']}")

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    print("[WARNING] FPS not detected, defaulting to 30")
    fps = 30.0

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

duration_sec = total_frames / fps

# Store in CONFIG for later use
CONFIG["FPS"] = fps
CONFIG["FRAME_WIDTH"] = width
CONFIG["FRAME_HEIGHT"] = height
CONFIG["TOTAL_FRAMES"] = total_frames

print(f"[INFO] Video Name       : {CONFIG['VIDEO_NAME']}")
print(f"[INFO] Total Frames     : {total_frames}")
print(f"[INFO] FPS              : {fps:.2f}")
print(f"[INFO] Resolution       : {width}x{height}")
print(f"[INFO] Duration (secs)  : {duration_sec:.2f}")
print(f"[INFO] Duration (mins)  : {duration_sec/60:.2f}")

# ⚠️ DO NOT release here if you will reuse cap later
# cap.release()

[INFO] Video Name       : 2.2.4
[INFO] Total Frames     : 2029
[INFO] FPS              : 29.97
[INFO] Resolution       : 3840x2160
[INFO] Duration (secs)  : 67.70
[INFO] Duration (mins)  : 1.13


In [775]:
# ------------------------------------------------------------
# 🧠 LOAD MODELS (FIXED)
# ------------------------------------------------------------
print("[INFO] Loading YOLO model...")
yolo = YOLO(CONFIG["YOLO_MODEL_PATH"])

print("[INFO] Loading TimeSformer model + labels...")
from transformers import AutoImageProcessor, TimesformerForVideoClassification
from sklearn.preprocessing import MultiLabelBinarizer

# ------------------------------------------------------------
# LOAD & CLEAN LABELS (FIXED PARSING)
# ------------------------------------------------------------
train_df = pd.read_csv(CONFIG["TRAIN_CSV"])

def parse_actions(action_str):
    # Convert string like "['walking','running']" → ['walking', 'running']
    action_str = str(action_str).strip().replace("[", "").replace("]", "").replace("'", "")
    return [a.strip() for a in action_str.split(",") if a.strip()]

all_actions_list = train_df["Actions"].apply(parse_actions).tolist()

label_encoder = MultiLabelBinarizer()
label_encoder.fit(all_actions_list)
action_labels = label_encoder.classes_

# ------------------------------------------------------------
# LOAD PROCESSOR + MODEL
# ------------------------------------------------------------
processor = AutoImageProcessor.from_pretrained(CONFIG["TSF_MODEL_PATH"])

tsf_model = TimesformerForVideoClassification.from_pretrained(
    CONFIG["TSF_MODEL_PATH"],
    num_labels=len(action_labels),
    ignore_mismatched_sizes=True
)

# ------------------------------------------------------------
# LOAD CHECKPOINT SAFELY
# ------------------------------------------------------------
checkpoint = torch.load(CONFIG["TSF_CHECKPOINT"], map_location=CONFIG["DEVICE"])

# Handle case where checkpoint has "model_state_dict"
if "model_state_dict" in checkpoint:
    checkpoint = checkpoint["model_state_dict"]

tsf_model.load_state_dict(checkpoint, strict=False)

# ------------------------------------------------------------
# DEVICE SETUP
# ------------------------------------------------------------
tsf_model.to(CONFIG["DEVICE"])
tsf_model.eval()

print(f"[INFO] Loaded TimeSformer model with {len(action_labels)} classes")

[INFO] Loading YOLO model...
[INFO] Loading TimeSformer model + labels...


Some weights of TimesformerForVideoClassification were not initialized from the model checkpoint at /home1/jtt_1/mtp/timesformer_base_patch16_224_k400 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([400, 768]) in the checkpoint and torch.Size([13, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([400]) in the checkpoint and torch.Size([13]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[INFO] Loaded TimeSformer model with 13 classes


In [776]:
# ------------------------------------------------------------
# 🎥 VIDEO SETUP + TRACKER INIT (FULL FIXED)
# ------------------------------------------------------------
import cv2

# Initialize video capture
cap = cv2.VideoCapture(CONFIG["VIDEO_PATH"])

if not cap.isOpened():
    raise RuntimeError(f"Could not open {CONFIG['VIDEO_PATH']}")

# -----------------------------
# Extract video properties
# -----------------------------
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    print("[WARNING] FPS not detected, defaulting to 30")
    fps = 30.0

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"[INFO] Video FPS: {fps:.2f}")
print(f"[INFO] Resolution: {width}x{height}")
print(f"[INFO] Total Frames: {total_frames}")

# -----------------------------
# Apply TEST MODE limit
# -----------------------------
if CONFIG["TEST_MODE"]:
    total_frames = min(CONFIG["TEST_MAX_FRAMES"], total_frames)
    print(f"[INFO] TEST_MODE enabled → limiting to {total_frames} frames")

# -----------------------------
# Store metadata globally
# -----------------------------
CONFIG["FPS"] = fps
CONFIG["FRAME_WIDTH"] = width
CONFIG["FRAME_HEIGHT"] = height
CONFIG["TOTAL_FRAMES"] = total_frames


[INFO] Video FPS: 29.97
[INFO] Resolution: 3840x2160
[INFO] Total Frames: 2029
[INFO] TEST_MODE enabled → limiting to 100 frames


In [777]:
import numpy as np
from scipy.optimize import linear_sum_assignment
import cv2

class KalmanFilter:
    def __init__(self):
        # State: [cx, cy, w, h, vx, vy, vw, vh]
        self.dt = 1.0

        self.F = np.eye(8)
        for i in range(4):
            self.F[i, i+4] = self.dt

        self.H = np.eye(4, 8)

        self.P = np.eye(8) * 10
        self.Q = np.eye(8)
        self.R = np.eye(4)

        self.x = None

    def initiate(self, bbox):
        x1, y1, x2, y2 = bbox
        w = x2 - x1
        h = y2 - y1
        cx = x1 + w / 2
        cy = y1 + h / 2

        self.x = np.array([cx, cy, w, h, 0, 0, 0, 0])

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.get_bbox()

    def update(self, bbox):
        x1, y1, x2, y2 = bbox
        w = x2 - x1
        h = y2 - y1
        cx = x1 + w / 2
        cy = y1 + h / 2

        z = np.array([cx, cy, w, h])

        y = z - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)

        self.x = self.x + K @ y
        self.P = (np.eye(8) - K @ self.H) @ self.P

    def get_bbox(self):
        cx, cy, w, h = self.x[:4]
        return [
            cx - w / 2,
            cy - h / 2,
            cx + w / 2,
            cy + h / 2
        ]

In [778]:
class DeepFeatureExtractor:
    def extract(self, frame, bbox):
        # Simple placeholder: color histogram
        x1, y1, x2, y2 = map(int, bbox)
        crop = frame[y1:y2, x1:x2]

        if crop.size == 0:
            return np.zeros(128)

        hist = cv2.calcHist([crop], [0, 1, 2], None, [8, 8, 8],
                            [0, 256, 0, 256, 0, 256])
        hist = cv2.normalize(hist, hist).flatten()

        return hist

In [779]:
def cosine_sim(a, b):
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return 0.0
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def iou(b1, b2):
    x1 = max(b1[0], b2[0])
    y1 = max(b1[1], b2[1])
    x2 = min(b1[2], b2[2])
    y2 = min(b1[3], b2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)

    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])

    return inter / (a1 + a2 - inter + 1e-6)


def center_distance(b1, b2):
    c1 = [(b1[0] + b1[2]) / 2, (b1[1] + b1[3]) / 2]
    c2 = [(b2[0] + b2[2]) / 2, (b2[1] + b2[3]) / 2]
    return np.linalg.norm(np.array(c1) - np.array(c2))

In [780]:
class Track:
    def __init__(self, bbox, tid, feature):
        self.id = tid
        self.kf = KalmanFilter()
        self.kf.initiate(bbox)

        self.feature = feature

        self.hits = 1
        self.lost = 0
        self.confirmed = False

        self.smooth_bbox = bbox
        self.last_box = bbox   # ✅ FIX

    def predict(self):
        return self.kf.predict()

    def update(self, bbox, feature):
        self.kf.update(bbox)

        alpha = 0.7
        new_bbox = self.kf.get_bbox()

        self.smooth_bbox = [
            alpha * new_bbox[i] + (1 - alpha) * self.smooth_bbox[i]
            for i in range(4)
        ]

        self.last_box = self.smooth_bbox   # ✅ FIX

        self.feature = 0.8 * self.feature + 0.2 * feature

        self.hits += 1
        self.lost = 0

        if self.hits >= 3:
            self.confirmed = True

In [781]:
import numpy as np
from scipy.optimize import linear_sum_assignment

class HybridTracker:
    def __init__(self, max_age=30, match_thresh=0.5):
        self.tracks = {}
        self.next_id = 0

        self.extractor = DeepFeatureExtractor()

        self.max_age = max_age
        self.match_thresh = match_thresh

    def update(self, detections, frame):
        outputs = []

        # -----------------------------
        # Extract features
        # -----------------------------
        det_feats = [self.extractor.extract(frame, d) for d in detections]

        # -----------------------------
        # INIT CASE
        # -----------------------------
        if len(self.tracks) == 0:
            for i, d in enumerate(detections):
                tid = self.next_id
                self.tracks[tid] = Track(d, tid, det_feats[i])

                # FIXED: return confirmed=False
                outputs.append((d, tid, False))

                self.next_id += 1
            return outputs

        track_ids = list(self.tracks.keys())
        preds = [self.tracks[t].predict() for t in track_ids]

        cost = np.ones((len(preds), len(detections)))

        # -----------------------------
        # COST MATRIX
        # -----------------------------
        for i, p in enumerate(preds):
            for j, d in enumerate(detections):

                # Gating (very important)
                if center_distance(p, d) > 200:
                    continue

                iou_score = iou(p, d)
                motion = np.exp(-center_distance(p, d) / 50)
                app = cosine_sim(self.tracks[track_ids[i]].feature, det_feats[j])

                sim = 0.25 * iou_score + 0.25 * motion + 0.5 * app
                cost[i, j] = 1 - sim

        r, c = linear_sum_assignment(cost)

        assigned = set()
        matched_tracks = set()

        # -----------------------------
        # MATCHED TRACKS
        # -----------------------------
        for i, j in zip(r, c):
            if (1 - cost[i, j]) > self.match_thresh:
                tid = track_ids[i]

                self.tracks[tid].update(detections[j], det_feats[j])

                # FIXED: include confirmed flag
                outputs.append((
                    self.tracks[tid].smooth_bbox,
                    tid,
                    self.tracks[tid].confirmed
                ))

                assigned.add(j)
                matched_tracks.add(tid)

        # -----------------------------
        # UNMATCHED TRACKS
        # -----------------------------
        for tid in track_ids:
            if tid not in matched_tracks:
                self.tracks[tid].lost += 1

        # -----------------------------
        # NEW TRACKS
        # -----------------------------
        for j, d in enumerate(detections):
            if j not in assigned:
                tid = self.next_id
                self.tracks[tid] = Track(d, tid, det_feats[j])

                # FIXED: confirmed=False
                outputs.append((d, tid, False))

                self.next_id += 1

        # -----------------------------
        # DELETE DEAD TRACKS
        # -----------------------------
        to_delete = []
        for tid, t in self.tracks.items():
            if t.lost > self.max_age or (not t.confirmed and t.lost > 5):
                to_delete.append(tid)

        for tid in to_delete:
            del self.tracks[tid]

        return outputs

In [782]:

# -----------------------------
# Initialize tracker
# -----------------------------
tracker = HybridTracker(max_age=CONFIG["MAX_AGE"], match_thresh=0.5)

print("[INFO] HybridTracker initialized successfully")

[INFO] HybridTracker initialized successfully


In [783]:
# ------------------------------------------------------------
# 🧰 UTILS (FIXED)
# ------------------------------------------------------------
def bbox_area_intersection_fraction(bbox, frame_w, frame_h):
    x1, y1, x2, y2 = bbox
    bx1, by1 = max(0, x1), max(0, y1)
    bx2, by2 = min(frame_w, x2), min(frame_h, y2)

    inter_w = max(0, bx2 - bx1)
    inter_h = max(0, by2 - by1)
    inter_area = inter_w * inter_h

    bbox_area = max(1.0, (x2 - x1) * (y2 - y1))
    return inter_area / bbox_area


def yolo_to_bboxes(result):
    try:
        boxes = result[0].boxes.xyxy.cpu().numpy()
    except:
        boxes = result.boxes.xyxy.cpu().numpy()

    bboxes = []
    for b in boxes:
        x1, y1, x2, y2 = map(float, b[:4])
        if x2 > x1 and y2 > y1:
            bboxes.append([x1, y1, x2, y2])
    return bboxes


def fraction_tracks_out_of_frame(tracker, frame_w, frame_h, threshold_inside=0.2):
    confirmed = [tr for tr in tracker.tracks.values() if tr.confirmed]
    if not confirmed:
        return 0.0

    out_count = 0
    for tr in confirmed:
        frac = bbox_area_intersection_fraction(tr.last_box, frame_w, frame_h)
        if frac < threshold_inside:
            out_count += 1

    return out_count / len(confirmed)


In [784]:
# ------------------------------------------------------------
# 🔁 MAIN LOOP (HYBRID TRACKER FIXED)
# ------------------------------------------------------------
from tqdm import tqdm
import time
from collections import deque

frame_buffers = {}
action_predictions = {}
results_rows = []

NUM_FRAMES = CONFIG["NUM_FRAMES"]
TOP_K = CONFIG["TOP_K"]

frame_idx = 0
start_time = time.time()
last_detect_frame = -999

print(f"[INFO] Using TimeSformer ({NUM_FRAMES} frames/clip, TOP_K={TOP_K})")

pbar = tqdm(total=CONFIG["TOTAL_FRAMES"], desc="Processing", unit="frame")
prev_time = time.time()

while frame_idx < CONFIG["TOTAL_FRAMES"]:
    ret, frame = cap.read()
    if not ret:
        break

    # -----------------------------
    # FPS
    # -----------------------------
    curr_time = time.time()
    loop_fps = 1.0 / (curr_time - prev_time + 1e-6)
    prev_time = curr_time

    # -----------------------------
    # DETECTION TRIGGER
    # -----------------------------
    do_detect = False

    if (frame_idx - last_detect_frame) >= CONFIG["DETECT_EVERY_N_FRAMES"]:
        do_detect = True

    if len(tracker.tracks) == 0:
        do_detect = True

    frac_out = fraction_tracks_out_of_frame(tracker, width, height)
    if frac_out >= CONFIG["OUT_OF_FRAME_FRACTION"]:
        do_detect = True

    detections = []

    # -----------------------------
    # YOLO DETECTION
    # -----------------------------
    if do_detect:
        yolo_res = yolo.predict(frame, conf=CONFIG["CONF_THRESH"], verbose=False)
        detections = yolo_to_bboxes(yolo_res)
        last_detect_frame = frame_idx

    # -----------------------------
    # TRACKER UPDATE
    # -----------------------------
    updates = tracker.update(detections, frame)

    active_ids = set()

    # -----------------------------
    # TRACK LOOP
    # -----------------------------
    for bbox, tid, confirmed in updates:

        # ❗ Ignore unconfirmed tracks (IMPORTANT)
        if not confirmed:
            continue

        x1, y1, x2, y2 = map(int, bbox)

        # Clip bbox
        x1 = max(0, min(width - 1, x1))
        y1 = max(0, min(height - 1, y1))
        x2 = max(0, min(width - 1, x2))
        y2 = max(0, min(height - 1, y2))

        if x2 <= x1 or y2 <= y1:
            continue

        crop = frame[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        active_ids.add(tid)

        # -----------------------------
        # BUFFER INIT
        # -----------------------------
        if tid not in frame_buffers:
            frame_buffers[tid] = deque(maxlen=NUM_FRAMES)

        frame_buffers[tid].append(cv2.resize(crop, (224, 224)))

        actions, scores = [], []

        # -----------------------------
        # TIMESFORMER
        # -----------------------------
        if len(frame_buffers[tid]) == NUM_FRAMES:

            clip = np.stack(frame_buffers[tid])

            inputs = processor(list(clip), return_tensors="pt")
            inputs = {k: v.to(CONFIG["DEVICE"]) for k, v in inputs.items()}

            with torch.no_grad():
                logits = tsf_model(**inputs).logits
                probs = torch.softmax(logits, dim=-1)[0]

                top = torch.topk(probs, k=TOP_K)

                actions = [action_labels[i] for i in top.indices.cpu().numpy()]
                scores = [float(p) for p in top.values.cpu().numpy()]

            action_predictions[tid] = (actions, scores)

        # -----------------------------
        # USE LAST PRED
        # -----------------------------
        if tid in action_predictions:
            actions, scores = action_predictions[tid]

        results_rows.append((
            frame_idx, tid,
            float(x1), float(y1), float(x2), float(y2),
            int(confirmed),
            ",".join(actions),
            ",".join([f"{s:.2f}" for s in scores])
        ))

    # -----------------------------
    # CLEAN DEAD TRACKS
    # -----------------------------
    dead_ids = set(frame_buffers.keys()) - active_ids

    for tid in dead_ids:
        frame_buffers.pop(tid, None)
        action_predictions.pop(tid, None)

    # -----------------------------
    # PROGRESS
    # -----------------------------
    remaining = CONFIG["TOTAL_FRAMES"] - frame_idx
    eta = remaining / (loop_fps + 1e-6)

    pbar.update(1)
    pbar.set_postfix({
        "frame": f"{frame_idx}/{CONFIG['TOTAL_FRAMES']}",
        "fps": f"{loop_fps:.2f}",
        "tracks": len(tracker.tracks),
        "ETA(s)": f"{eta:.1f}"
    })

    frame_idx += 1

pbar.close()

# -----------------------------
# SAVE
# -----------------------------
elapsed = time.time() - start_time
proc_fps = frame_idx / elapsed if elapsed > 0 else 0.0

print(f"[INFO] Processed {frame_idx} frames in {elapsed:.2f}s → {proc_fps:.2f} FPS")

cap.release()

df = pd.DataFrame(results_rows, columns=[
    "frame_idx", "track_id", "x1", "y1", "x2", "y2",
    "confirmed", "actions", "scores"
])

df = df.drop_duplicates(subset=["frame_idx", "track_id"], keep="last")
df.to_csv(CONFIG["RESULTS_CSV"], index=False)

print(f"[INFO] Saved results → {CONFIG['RESULTS_CSV']}")

[INFO] Using TimeSformer (16 frames/clip, TOP_K=1)


Processing: 100%|██████████| 100/100 [00:17<00:00,  5.81frame/s, frame=99/100, fps=51.79, tracks=1, ETA(s)=0.0]


[INFO] Processed 100 frames in 17.20s → 5.81 FPS
[INFO] Saved results → /home1/jtt_1/mtp/outputs-with-deep-sort-tracking-single/2.2.4/2.2.4_tracking.csv


In [785]:
# Add before render loop
prev_time = time.time()
fps_display = 0.0

def draw_confidence_bar(frame, x, y, width, height, score, color):
    """Draw horizontal confidence bar"""
    # Background
    cv2.rectangle(frame, (x, y), (x + width, y + height), (50, 50, 50), -1)

    # Filled portion
    filled_w = int(width * score)
    cv2.rectangle(frame, (x, y), (x + filled_w, y + height), color, -1)

    # Border
    cv2.rectangle(frame, (x, y), (x + width, y + height), (255, 255, 255), 1)


def draw_metrics_overlay(frame, fps_display, num_tracks):
    """Top-left overlay for FPS and stats"""
    overlay_texts = [
        f"FPS: {fps_display:.1f}",
        f"Active Tracks: {num_tracks}"
    ]

    x, y = 20, 40
    for text in overlay_texts:
        cv2.putText(frame, text, (x+2, y+2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                    (0,0,0), 3, cv2.LINE_AA)
        cv2.putText(frame, text, (x, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                    (0,255,0), 2, cv2.LINE_AA)
        y += 35

In [786]:
# ------------------------------------------------------------
# 🎥 ANNOTATED VIDEO RENDERING (FULL FIXED)
# ------------------------------------------------------------
import cv2
import pandas as pd
import os
import random
from tqdm import tqdm
import numpy as np

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
VIDEO_PATH = CONFIG["VIDEO_PATH"]
VIDEO_NAME = CONFIG["VIDEO_NAME"]
OUTPUT_DIR = CONFIG["OUTPUT_DIR"]

CSV_PATH = f"{OUTPUT_DIR}/{VIDEO_NAME}_tracking.csv"
OUTPUT_VIDEO = f"{OUTPUT_DIR}/{VIDEO_NAME}_output.mp4"

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    print("[WARNING] FPS not detected, defaulting to 30")
    fps = 30.0

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (w, h))

total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if CONFIG.get("TEST_MODE", False):
    total = min(CONFIG.get("TEST_MAX_FRAMES", 200), total)

print(f"[INFO] Rendering {total} frames...")

# ------------------------------------------------------------
# FONT SETTINGS (DYNAMIC)
# ------------------------------------------------------------
FONT_SCALE = max(0.6, min(1.4, h / 720 * 0.8))
THICKNESS = int(max(2, h / 720 * 2))
PADDING_X = int(10 * (h / 720))
PADDING_Y = int(6 * (h / 720))

# ------------------------------------------------------------
# COLOR HELPERS
# ------------------------------------------------------------
track_colors = {}

def get_color(tid):
    if tid not in track_colors:
        random.seed(tid)
        track_colors[tid] = tuple(random.randint(40, 200) for _ in range(3))
    return track_colors[tid]

def lighten_color(color, factor=1.5):
    return tuple(min(255, int(c * factor)) for c in color)

# ------------------------------------------------------------
# DRAW HELPERS
# ------------------------------------------------------------
def draw_filled_rounded_rect(img, x1, y1, x2, y2, color, radius=8, alpha=0.9):
    overlay = img.copy()
    cv2.rectangle(overlay, (x1 + radius, y1), (x2 - radius, y2), color, -1)
    cv2.rectangle(overlay, (x1, y1 + radius), (x2, y2 - radius), color, -1)

    cv2.circle(overlay, (x1 + radius, y1 + radius), radius, color, -1)
    cv2.circle(overlay, (x2 - radius, y1 + radius), radius, color, -1)
    cv2.circle(overlay, (x1 + radius, y2 - radius), radius, color, -1)
    cv2.circle(overlay, (x2 - radius, y2 - radius), radius, color, -1)

    cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0, img)

def draw_label(frame, text, pos, color):
    font = cv2.FONT_HERSHEY_SIMPLEX
    x, y = pos

    (tw, th), _ = cv2.getTextSize(text, font, FONT_SCALE, THICKNESS)

    # Clamp box
    x1 = max(0, x - PADDING_X)
    y1 = max(0, y - th - PADDING_Y)
    x2 = min(frame.shape[1] - 1, x + tw + PADDING_X)
    y2 = min(frame.shape[0] - 1, y + PADDING_Y)

    bg_color = lighten_color(color, 1.5)
    draw_filled_rounded_rect(frame, x1, y1, x2, y2, bg_color)

    # Text with outline
    cv2.putText(frame, text, (x + 2, y - 2), font, FONT_SCALE,
                (0, 0, 0), THICKNESS + 2, cv2.LINE_AA)
    cv2.putText(frame, text, (x + 2, y - 2), font, FONT_SCALE,
                (255, 255, 255), THICKNESS, cv2.LINE_AA)

# ------------------------------------------------------------
# PRE-GROUP (FASTER)
# ------------------------------------------------------------
frame_dict = {}
for _, row in df.iterrows():
    f = int(row.frame_idx)
    if f not in frame_dict:
        frame_dict[f] = []
    frame_dict[f].append(row)

# ------------------------------------------------------------
# RENDER LOOP
# ------------------------------------------------------------
pbar = tqdm(total=total, desc="[Rendering Video]", unit="frame")

for i in range(total):
    ret, frame = cap.read()
    if not ret:
        break

    # FPS calculation
    curr_time = time.time()
    fps_display = 1.0 / (curr_time - prev_time + 1e-6)
    prev_time = curr_time

    num_tracks = 0

    if i in frame_dict:
        for r in frame_dict[i]:

            x1 = int(max(0, min(w - 1, r.x1)))
            y1 = int(max(0, min(h - 1, r.y1)))
            x2 = int(max(0, min(w - 1, r.x2)))
            y2 = int(max(0, min(h - 1, r.y2)))

            if x2 <= x1 or y2 <= y1:
                continue

            tid = int(r.track_id)
            actions = str(r.actions).strip()
            scores = str(r.scores).strip()

            color = get_color(tid)
            num_tracks += 1

            # Draw bbox
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, THICKNESS)

            draw_label(frame, f"ID {tid}", (x1, max(20, y1 - 10)), color)

            # ----------------------------
            # CONFIDENCE BAR + ACTION TEXT
            # ----------------------------
            if actions.lower() not in ["nan", "none", "null", ""]:

                act_list = [a.strip() for a in actions.split(",") if a.strip()]
                score_list = [float(s) for s in scores.split(",")] if scores not in ["nan", ""] else []

                for idx, act in enumerate(act_list):
                    score = score_list[idx] if idx < len(score_list) else 0.0

                    label_y = y2 + 20 + idx * 30
                    if label_y > h - 10:
                        label_y = y1 - 40 - idx * 30

                    # Draw label
                    draw_label(frame, f"{act} ({score:.2f})", (x1, label_y), color)

                    # Draw confidence bar
                    draw_confidence_bar(
                        frame,
                        x1,
                        label_y + 5,
                        width=120,
                        height=8,
                        score=score,
                        color=color
                    )

    # Draw FPS + metrics
    draw_metrics_overlay(frame, fps_display, num_tracks)

    out.write(frame)
    pbar.update(1)

pbar.close()
cap.release()
out.release()

print(f"[INFO] Annotated video saved → {OUTPUT_VIDEO}")

[INFO] Rendering 100 frames...


[Rendering Video]:   0%|          | 0/100 [00:00<?, ?frame/s]

[Rendering Video]: 100%|██████████| 100/100 [00:04<00:00, 20.65frame/s]

[INFO] Annotated video saved → /home1/jtt_1/mtp/outputs-with-deep-sort-tracking-single/2.2.4/2.2.4_output.mp4


In [787]:
# ------------------------------------------------------------
# 📊 EVALUATION: GT vs PRED CSV
# ------------------------------------------------------------
import pandas as pd
import numpy as np
from collections import defaultdict

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
PRED_FILE = CONFIG['RESULTS_CSV']


In [788]:

# ------------------------------------------------------------
# IOU FUNCTION
# ------------------------------------------------------------
def compute_iou(b1, b2):
    x1 = max(b1[0], b2[0])
    y1 = max(b1[1], b2[1])
    x2 = min(b1[2], b2[2])
    y2 = min(b1[3], b2[3])

    inter = max(0, x2-x1) * max(0, y2-y1)

    a1 = (b1[2]-b1[0])*(b1[3]-b1[1])
    a2 = (b2[2]-b2[0])*(b2[3]-b2[1])

    return inter / (a1 + a2 - inter + 1e-6)



# ------------------------------------------------------------
# PRED DEDUP
# ------------------------------------------------------------
def remove_pred_duplicates(boxes, iou_thresh=CONFIG['IOU_THRESH']):
    keep = []
    for b in boxes:
        if all(compute_iou(b["bbox"], k["bbox"]) < iou_thresh for k in keep):
            keep.append(b)
    return keep

def deduplicate_boxes_cluster(boxes, iou_thresh=CONFIG['IOU_THRESH']):
    clusters = []

    for b in boxes:
        added = False

        for cluster in clusters:
            # compare with cluster representative
            if compute_iou(b["bbox"], cluster[0]["bbox"]) > iou_thresh:
                cluster.append(b)
                added = True
                break

        if not added:
            clusters.append([b])

    # take one box per cluster
    dedup_boxes = [cluster[0] for cluster in clusters]
    return dedup_boxes

# ------------------------------------------------------------
# LOAD GT
# ------------------------------------------------------------
gt_frames = defaultdict(list)

with open(GT_FILE) as f:
    for line in f:
        parts = line.strip().split()

        if len(parts) < 6:
            continue

        frame = int(parts[0])

        if  CONFIG['TEST_MODE'] and frame >= CONFIG['TEST_MAX_FRAMES']:
            continue

        x1, y1, x2, y2 = map(float, parts[1:5])
        action = parts[-1].replace('"', '')

        gt_frames[frame].append({
            "bbox": [x1, y1, x2, y2],
            "action": action
        })

def filter_small_boxes(boxes, min_area=5000):
    filtered = []
    for b in boxes:
        x1, y1, x2, y2 = b["bbox"]
        area = (x2 - x1) * (y2 - y1)
        if area >= min_area:
            filtered.append(b)
    return filtered
# ------------------------------------------------------------
# APPLY GT DEDUP (FIXED)
# ------------------------------------------------------------
gt_frames_dedup = {}

for frame, boxes in gt_frames.items():
    if len(boxes) == 0:
        continue

    dedup_boxes = deduplicate_boxes_cluster(boxes, iou_thresh=CONFIG['IOU_THRESH'])
    gt_frames_dedup[frame] = dedup_boxes


# Debug GT stats
gt_counts = [len(v) for v in gt_frames_dedup.values() if len(v) > 0]

if len(gt_counts) > 0:
    print("[INFO] Avg GT per frame AFTER dedup:", np.mean(gt_counts))
    print("Avg GT AFTER size filter:",
      np.mean([len(filter_small_boxes(v)) for v in gt_frames_dedup.values()]))
else:
    print("[WARNING] No GT boxes after dedup!")


# ------------------------------------------------------------
# LOAD PRED
# ------------------------------------------------------------
pred_df = pd.read_csv(PRED_FILE)

FRAME_OFFSET = int(pred_df.frame_idx.min()) - int(min(gt_frames_dedup.keys()))
print(f"[INFO] Auto Frame Offset = {FRAME_OFFSET}")

pred_frames = defaultdict(list)

for _, r in pred_df.iterrows():

    frame = int(r.frame_idx) - FRAME_OFFSET

    if  CONFIG['TEST_MODE'] and frame >= CONFIG['TEST_MAX_FRAMES']:
        continue

    if frame < 0:
        continue

    if int(r.confirmed) != 1:
        continue

    if pd.isna(r.x1) or pd.isna(r.y1):
        continue

    action = ""
    if pd.notna(r.actions):
        action = str(r.actions).split(",")[0]

    pred_frames[frame].append({
        "bbox": [r.x1, r.y1, r.x2, r.y2],
        "action": action
    })


# ------------------------------------------------------------
# DEBUG CHECK
# ------------------------------------------------------------
common_frames = set(gt_frames_dedup.keys()) & set(pred_frames.keys())
print(f"[INFO] Common frames: {len(common_frames)}")

if len(common_frames) > 0:
    f = list(common_frames)[0]
    print("[DEBUG] Sample GT box:", gt_frames_dedup[f][0]["bbox"])
    print("[DEBUG] Sample Pred box:", pred_frames[f][0]["bbox"])
    print("[DEBUG] Sample IoU:", compute_iou(
        gt_frames_dedup[f][0]["bbox"],
        pred_frames[f][0]["bbox"]
    ))


# ------------------------------------------------------------
# MATCHING
# ------------------------------------------------------------
TP = 0
FP = 0
FN = 0

correct_action = 0
total_matched = 0
total_iou = 0

all_frames = sorted(gt_frames_dedup.keys())

for frame in all_frames:

    gt_list = filter_small_boxes(gt_frames_dedup.get(frame, []), min_area=5000)
    pred_list = pred_frames.get(frame, [])

    # Dedup predictions
    pred_list = remove_pred_duplicates(pred_list)

    matched_gt = set()

    for pred in pred_list:
        best_iou = 0
        best_j = -1

        for j, gt in enumerate(gt_list):
            if j in matched_gt:
                continue

            iou = compute_iou(pred["bbox"], gt["bbox"])
            if iou > best_iou:
                best_iou = iou
                best_j = j

        if best_iou >= CONFIG['IOU_THRESH']:
            TP += 1
            matched_gt.add(best_j)

            total_matched += 1
            total_iou += best_iou

            if pred["action"] == gt_list[best_j]["action"]:
                correct_action += 1
        else:
            FP += 1

    FN += len(gt_list) - len(matched_gt)


# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------
precision = TP / (TP + FP + 1e-6)
recall = TP / (TP + FN + 1e-6)
f1 = 2 * precision * recall / (precision + recall + 1e-6)
action_acc = correct_action / (total_matched + 1e-6)
mean_iou = total_iou / (total_matched + 1e-6)


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------
print("\n===== FINAL EVALUATION =====")
print(f"Frames evaluated: {len(all_frames)}")
print(f"TP: {TP}")
print(f"FP: {FP}")
print(f"FN: {FN}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Mean IoU: {mean_iou:.4f}")

[INFO] Avg GT per frame AFTER dedup: 22.59090909090909
Avg GT AFTER size filter: 18.803030303030305
[INFO] Auto Frame Offset = 2
[INFO] Common frames: 66
[DEBUG] Sample GT box: [2105.0, 941.0, 2220.0, 1093.0]
[DEBUG] Sample Pred box: [1471.0, 255.0, 1567.0, 346.0]
[DEBUG] Sample IoU: 0.0

===== FINAL EVALUATION =====
Frames evaluated: 66
TP: 5
FP: 194
FN: 1236
Precision: 0.0251
Recall: 0.0040
F1 Score: 0.0069
Mean IoU: 0.6397


In [789]:
# END

In [790]:
# # ------------------------------------------------------------
# # 🎯 ADD: TRACK TRAILS (MOTION PATHS)
# # ------------------------------------------------------------

# from collections import deque

# # Store track history
# track_history = {}   # tid -> deque of points

# MAX_TRAIL_LENGTH = 30


# def draw_trail(frame, points, color):
#     """Draw motion trail"""
#     for i in range(1, len(points)):
#         if points[i-1] is None or points[i] is None:
#             continue

#         thickness = int(2 + 4 * (i / len(points)))  # tail thinner
#         cv2.line(frame, points[i-1], points[i], color, thickness)


# # ------------------------------------------------------------
# # 🔁 MODIFY INSIDE YOUR RENDER LOOP
# # ------------------------------------------------------------

# for i in range(total):
#     ret, frame = cap.read()
#     if not ret:
#         break

#     if i in frame_dict:
#         for r in frame_dict[i]:

#             x1 = int(max(0, min(w - 1, r.x1)))
#             y1 = int(max(0, min(h - 1, r.y1)))
#             x2 = int(max(0, min(w - 1, r.x2)))
#             y2 = int(max(0, min(h - 1, r.y2)))

#             if x2 <= x1 or y2 <= y1:
#                 continue

#             tid = int(r.track_id)
#             color = get_color(tid)

#             # ----------------------------
#             # TRACK CENTER
#             # ----------------------------
#             cx = int((x1 + x2) / 2)
#             cy = int((y1 + y2) / 2)

#             if tid not in track_history:
#                 track_history[tid] = deque(maxlen=MAX_TRAIL_LENGTH)

#             track_history[tid].append((cx, cy))

#             # ----------------------------
#             # DRAW TRAIL
#             # ----------------------------
#             draw_trail(frame, track_history[tid], color)

#             # Draw bbox
#             cv2.rectangle(frame, (x1, y1), (x2, y2), color, THICKNESS)

#             draw_label(frame, f"ID {tid}", (x1, max(20, y1 - 10)), color)

#     out.write(frame)